##### Experiment V2: Fine-Tuned ResNet50

##### Objective

Baseline (V1):

- Test Accuracy = 92.4%
- Macro F1 = 0.93
- Glioma Recall = 0.80

Problem:

The model confuses glioma and meningioma images.

Proposed Solution:

Fine-tune the last layers of ResNet50 using a very small learning rate.

Success Criteria:

- Improve Glioma Recall
- Maintain or improve Test Accuracy
- Reduce Glioma → Meningioma confusion

In [ ]:
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

tf.keras.utils.set_random_seed(42)

In [2]:
DATASET_DIR = Path(
    "../artifacts/data_ingestion/brisc2025/brisc2025/classification_task"
)

train_dir = DATASET_DIR / "train"
test_dir = DATASET_DIR / "test"

print(train_dir)
print(test_dir)

..\artifacts\data_ingestion\brisc2025\brisc2025\classification_task\train
..\artifacts\data_ingestion\brisc2025\brisc2025\classification_task\test


#### TRAIN/TEST/VALIDATION SPLIT

In [3]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(224,224),
    batch_size=16
)

Found 5000 files belonging to 4 classes.
Using 4000 files for training.


In [4]:


val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224,224),
    batch_size=16
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(224,224),
    batch_size=16,
    shuffle=False
)

Found 5000 files belonging to 4 classes.
Using 1000 files for validation.
Found 1000 files belonging to 4 classes.


##### Build Fine-Tuning Model

In [5]:
base_model = tf.keras.applications.ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

In [6]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

In [7]:
trainable_layers = sum(
    layer.trainable
    for layer in base_model.layers
)

print(
    f"Trainable Layers: {trainable_layers}"
)

Trainable Layers: 30


#### Dataset Optimization

In [8]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

#### Fine-Tuned Model

In [9]:
model = tf.keras.Sequential([

    tf.keras.layers.Input(
        shape=(224,224,3)
    ),

    tf.keras.layers.Lambda(
        tf.keras.applications.resnet50.preprocess_input
    ),

    base_model,

    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Dense(
        256,
        activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(
            0.001
        )
    ),

    tf.keras.layers.Dropout(
        0.3
    ),

    tf.keras.layers.Dense(
        4,
        activation="softmax"
    )
])

In [10]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,121,476 (92.02 MB)

 Trainable params: 14,979,844 (57.14 MB)

 Non-trainable params: 9,141,632 (34.87 MB)

In [11]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

#### Callbacks

In [12]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    verbose=1
)

#### Train

In [13]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[
        early_stopping,
        reduce_lr
    ]
)

Epoch 1/15


c:\Projects\brain_tumor_detection\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


250/250 ━━━━━━━━━━━━━━━━━━━━ 480s 2s/step - accuracy: 0.6622 - loss: 1.3661 - val_accuracy: 0.8890 - val_loss: 0.7929 - learning_rate: 1.0000e-05
Epoch 2/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 475s 2s/step - accuracy: 0.8855 - loss: 0.7661 - val_accuracy: 0.9390 - val_loss: 0.6471 - learning_rate: 1.0000e-05
Epoch 3/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 483s 2s/step - accuracy: 0.9420 - loss: 0.6296 - val_accuracy: 0.9510 - val_loss: 0.6063 - learning_rate: 1.0000e-05
Epoch 4/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 441s 2s/step - accuracy: 0.9592 - loss: 0.5743 - val_accuracy: 0.9550 - val_loss: 0.5873 - learning_rate: 1.0000e-05
Epoch 5/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 478s 2s/step - accuracy: 0.9758 - loss: 0.5398 - val_accuracy: 0.9580 - val_loss: 0.5738 - learning_rate: 1.0000e-05
Epoch 6/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 461s 2s/step - accuracy: 0.9852 - loss: 0.5035 - val_accuracy: 0.9580 - val_loss: 0.5602 - learning_rate: 1.0000e-05
Epoch 7/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 472s 2s/step - accuracy: 0.99

In [15]:
save_dir = Path("../artifacts/model_trainer")
save_dir.mkdir(parents=True, exist_ok=True)

model.save(
    save_dir / "resnet50_finetuned.keras"
)

In [16]:
history_df = pd.DataFrame(
    history.history
)

history_df.to_csv(
    save_dir / "finetune_history.csv",
    index=False
)

NameError: name 'pd' is not defined